# NutriPrompt Technical Demo v2

Versión preparada para ejecutarse en VS Code + Jupyter local.

# Objetivo del notebook

Este notebook sirve para:
- probar integración con Gemini y OpenAI
- entender Jupyter local
- experimentar prompts
- validar outputs JSON
Los planes generados son orientativos y no sustituyen el asesoramiento de un profesional nutricional.

## 1. Instalación de dependencias

In [1]:
%pip install -q google-genai openai python-dotenv pandas ipywidgets


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Imports y entorno

In [2]:
import os
from dotenv import load_dotenv
from google import genai
import pandas as pd
import json

load_dotenv()

print("✅ Librerías cargadas correctamente")


✅ Librerías cargadas correctamente


## 3. Configuración Gemini

In [3]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError("No se ha encontrado GOOGLE_API_KEY en el archivo .env")

gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

print("✅ Conexión con Gemini preparada correctamente")


✅ Conexión con Gemini preparada correctamente


## 4. Datos del cliente

In [4]:
datos_cliente = {
    "nombre": "Beatriz",
    "objetivo": "Mejorar digestión",
    "restricciones": ["Low FODMAP", "Sin lactosa"],
    "actividad": "Moderada",
    "presupuesto": "Medio",
    "tiempo_cocina": "30 minutos"
}

datos_cliente


{'nombre': 'Beatriz',
 'objetivo': 'Mejorar digestión',
 'restricciones': ['Low FODMAP', 'Sin lactosa'],
 'actividad': 'Moderada',
 'presupuesto': 'Medio',
 'tiempo_cocina': '30 minutos'}

## 5. Prompt del sistema

In [5]:
SYSTEM_PROMPT = """
Eres un nutricionista experto.

Devuelve SIEMPRE:
- JSON válido
- 3 días de planificación
- desayuno
- comida
- cena
- ejercicio
- descanso
- coste_estimado

No añadas explicaciones fuera del JSON.
"""

print("✅ SYSTEM_PROMPT preparado")


✅ SYSTEM_PROMPT preparado


## 6. Prompt del usuario

In [6]:
USER_PROMPT = f"""
Cliente:
{json.dumps(datos_cliente, indent=2, ensure_ascii=False)}
"""

print(USER_PROMPT)



Cliente:
{
  "nombre": "Beatriz",
  "objetivo": "Mejorar digestión",
  "restricciones": [
    "Low FODMAP",
    "Sin lactosa"
  ],
  "actividad": "Moderada",
  "presupuesto": "Medio",
  "tiempo_cocina": "30 minutos"
}



## 7. Prompt completo

In [7]:
full_prompt = f"""
{SYSTEM_PROMPT}

{USER_PROMPT}
"""

print("✅ Prompt completo generado")


✅ Prompt completo generado


## 8. Generación con Gemini

In [8]:
try:
    response = gemini_client.models.generate_content(
        model="gemini-2.0-flash",
        contents=full_prompt
    )

    raw_output = response.text

    print("✅ Respuesta recibida\n")
    print(raw_output)

except Exception as e:
    print("❌ Error al llamar a Gemini")
    print("Tipo de error:", type(e).__name__)
    print("Detalle:")
    print(e)


❌ Error al llamar a Gemini
Tipo de error: ClientError
Detalle:
429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 25.47371667s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/ge

## 9. Limpieza JSON

In [9]:
def limpiar_salida_json(texto):
    return (
        texto
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )

clean_output = limpiar_salida_json(raw_output)

print(clean_output)


NameError: name 'raw_output' is not defined

## 10. Parsear JSON

In [ ]:
try:
    plan_data = json.loads(clean_output)
    print("✅ JSON válido")
    print(f"Número de días generados: {len(plan_data)}")
except json.JSONDecodeError as e:
    print("❌ Error al parsear JSON")
    print(e)


## 11. DataFrame

In [ ]:
df = pd.DataFrame(plan_data)

df


In [ ]:
from openai import OpenAI

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("No se encontró OPENAI_API_KEY")

openai_client = OpenAI(api_key=OPENAI_API_KEY)

print("✅ OpenAI preparado")

✅ OpenAI preparado


In [ ]:
response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": USER_PROMPT
        }
    ]
)

gpt_output = response.choices[0].message.content

print("✅ Respuesta GPT recibida\n")
print(gpt_output)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}